# Beyond GDP: Human Development and Capability Analysis

## Project overview

GDP per capita is the most common way to measure how well a country is doing.
But a high GDP does not always mean people live longer, are better educated, feel
free, or trust their institutions. This project asks a simple question:

**Is GDP alone enough to explain human development and well-being, or do health,
education, freedom, social support and governance matter as well?**

The framing comes from Amartya Sen's capability approach, which argues that
development should be judged by what people are actually able to do and be, not
just by income.

To study this, we bring together four sources for a common reference year (2023):

- UNDP Human Development Report 2025 — HDI, life expectancy, schooling, GNI
- World Happiness Report — happiness score and related factors
- World Bank — GDP per capita, health expenditure, unemployment, inflation, population
- Transparency International — Corruption Perceptions Index

We combine these into one country-level dataset and then explore how income
relates to the other dimensions of development, using SQL, statistics, clustering,
and a simple experimental capability index inspired by Sen.

## Notebook 1 — Data Understanding

This notebook covers the first stage only: loading each source, checking its
structure, coverage and missing values, and preparing clean tables for the
cleaning and analysis steps that follow.

Steps:
1. Setup and data path
2. UNDP HDI data
3. World Happiness data
4. World Bank data
5. Corruption (CPI) data
6. Coverage and missing-value summary

## 1. Setup and data path

We import the libraries used in this notebook and set the path to the raw data
folder. The full path is used so the notebook works regardless of where Jupyter
is launched from.

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_RAW = Path(r"D:\beyond-gdp-capability-analysis\data\raw")

print("Data folder found:", DATA_RAW.exists())

Data folder found: True


## 2. UNDP HDI data

Source: Human Development Report 2025 statistical annex, Table 1 (reference year 2023).

The table has a header spread across three rows and blank spacer columns between
the values, so we first load it without a header to see the raw layout, then pick
the columns we need.

In [12]:
hdi_file = DATA_RAW / "HDR25_Statistical_Annex_HDI_Table.xlsx"
hdi_raw = pd.read_excel(hdi_file, sheet_name="Table 1. HDI", header=None)

print("Shape:", hdi_raw.shape)
hdi_raw.iloc[4:12, 0:11]

Shape: (279, 15)


,0,1,2,3,4,5,6,7,8,9,10
4,NaN,NaN,Human Development Index (HDI),NaN,Life expectancy at birth,NaN,Expected years of schooling,NaN,Mean years of schooling,NaN,Gross national income (GNI) per capita
5,HDI rank,Country,Value,NaN,(years),NaN,(years),NaN,(years),NaN,(2021 PPP $)
6,NaN,NaN,2023,NaN,2023,NaN,2023,a,2023,a,2023
7,NaN,Very high human development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,Iceland,0.972,NaN,82.691,NaN,18.85059,c,13.908926,d,69116.93736
9,2,Norway,0.97,NaN,83.308,NaN,18.79285,c,13.117962,e,112710.0211
10,2,Switzerland,0.97,NaN,83.954,NaN,16.66753,NaN,13.949121,e,81948.90177
11,4,Denmark,0.962,NaN,81.933,NaN,18.70401,c,13.027321,e,76007.85669


The values sit in columns 0, 1, 2, 4, 6, 8 and 10. Only real countries have a
number in the rank column, while group labels like "Very high human development"
do not. We use that to keep countries and drop the labels, then rename the columns
and convert the values to numbers.

In [13]:
cols = {
    0: "hdi_rank",
    1: "country",
    2: "hdi",
    4: "life_expectancy",
    6: "expected_schooling",
    8: "mean_schooling",
    10: "gni_per_capita",
}

hdi = hdi_raw[list(cols)].rename(columns=cols)
hdi = hdi[pd.to_numeric(hdi["hdi_rank"], errors="coerce").notna()].copy()

for c in hdi.columns:
    if c != "country":
        hdi[c] = pd.to_numeric(hdi[c], errors="coerce")

hdi = hdi.reset_index(drop=True)

print("Shape:", hdi.shape)
print(hdi.isna().sum())
hdi.head()

Shape: (193, 7)
hdi_rank              0
country               0
hdi                   0
life_expectancy       0
expected_schooling    0
mean_schooling        0
gni_per_capita        0
dtype: int64


,hdi_rank,country,hdi,life_expectancy,expected_schooling,mean_schooling,gni_per_capita
0,1,Iceland,0.972,82.691,18.850590,13.908926,69116.93736
1,2,Norway,0.970,83.308,18.792850,13.117962,112710.02110
2,2,Switzerland,0.970,83.954,16.667530,13.949121,81948.90177
3,4,Denmark,0.962,81.933,18.704010,13.027321,76007.85669
4,5,Germany,0.959,81.378,17.309219,14.296372,64053.22124


## 3. World Happiness data

Source: World Happiness Report (Figure 2.1 data, years 2011 to 2025).

This file has one normal header row. We load it, look at the columns, then keep
only the 2023 rows to match the project's reference year. The social support,
freedom, generosity and corruption columns are "explained by" contributions to
the happiness score, not raw survey values, so we name them as contributions.

In [14]:
whr_file = DATA_RAW / "WHR26_Data_Figure_2_1.xlsx"
whr_raw = pd.read_excel(whr_file)

print("Shape:", whr_raw.shape)
print(whr_raw.columns.tolist())
whr_raw.head()

Shape: (2116, 13)
['Year', 'Rank', 'Country name', 'Life evaluation (3-year average)', 'Lower whisker', 'Upper whisker', 'Explained by: Log GDP per capita', 'Explained by: Social support', 'Explained by: Healthy life expectancy', 'Explained by: Freedom to make life choices', 'Explained by: Generosity', 'Explained by: Perceptions of corruption', 'Dystopia + residual']


,Year,Rank,Country name,Life evaluation (3-year average),Lower whisker,Upper whisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,2025,1,Finland,7.764,7.690,7.837,1.915,1.638,0.939,1.105,0.093,0.491,1.582
1,2025,2,Iceland,7.540,7.449,7.630,1.971,1.720,0.996,1.105,0.187,0.187,1.373
2,2025,3,Denmark,7.539,7.446,7.631,1.986,1.633,0.930,1.081,0.125,0.474,1.310
3,2025,4,Costa Rica,7.439,7.356,7.522,1.697,1.483,0.739,1.101,0.059,0.122,2.236
4,2025,5,Sweden,7.255,7.172,7.337,1.950,1.570,1.027,1.070,0.149,0.447,1.041


We keep only the 2023 rows and select the columns we need: happiness score and the
contribution columns. We rename them to short names, and add a "_contrib" suffix to
the explained-by columns so it is clear these are contributions, not raw values.

In [15]:
whr_2023 = whr_raw[whr_raw["Year"] == 2023].copy()

whr_cols = {
    "Country name": "country",
    "Life evaluation (3-year average)": "happiness_score",
    "Explained by: Social support": "social_support_contrib",
    "Explained by: Freedom to make life choices": "freedom_contrib",
    "Explained by: Generosity": "generosity_contrib",
    "Explained by: Perceptions of corruption": "corruption_contrib",
}

happiness = whr_2023[list(whr_cols)].rename(columns=whr_cols).reset_index(drop=True)

print("Shape:", happiness.shape)
print(happiness.isna().sum())
happiness.head()

Shape: (143, 6)
country                   0
happiness_score           0
social_support_contrib    3
freedom_contrib           3
generosity_contrib        3
corruption_contrib        3
dtype: int64


,country,happiness_score,social_support_contrib,freedom_contrib,generosity_contrib,corruption_contrib
0,Finland,7.741,1.572,0.859,0.142,0.546
1,Denmark,7.583,1.520,0.823,0.204,0.548
2,Iceland,7.525,1.617,0.819,0.258,0.182
3,Sweden,7.344,1.501,0.838,0.221,0.524
4,Israel,7.341,1.513,0.641,0.153,0.193


In [16]:
happiness[happiness["social_support_contrib"].isna()]

,country,happiness_score,social_support_contrib,freedom_contrib,generosity_contrib,corruption_contrib
61,Bahrain,5.959,NaN,NaN,NaN,NaN
87,Tajikistan,5.281,NaN,NaN,NaN,NaN
102,State of Palestine,4.879,NaN,NaN,NaN,NaN


## 4. World Bank data

Source: World Bank (five indicators in separate CSV files): GDP per capita, health
expenditure, unemployment, inflation, population.

These files have four information lines at the top before the real header, and one
column per year from 1960 to 2025. We first open one file to confirm the layout,
then load all five and pull out the 2023 value for each country.

In [17]:
gdp_file = DATA_RAW / "API_NY_GDP_PCAP_CD_DS2_en_csv_v2_358064.csv"
gdp_raw = pd.read_csv(gdp_file, skiprows=4)

print("Shape:", gdp_raw.shape)
print(gdp_raw.columns[:6].tolist())
print(gdp_raw.columns[-3:].tolist())
gdp_raw[["Country Name", "Country Code", "2023"]].head()

Shape: (266, 71)
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961']
['2024', '2025', 'Unnamed: 70']


,Country Name,Country Code,2023
0,Aruba,ABW,35718.753119
1,Africa Eastern and Southern,AFE,1571.449189
2,Afghanistan,AFG,413.757895
3,Africa Western and Central,AFW,1841.855064
4,Angola,AGO,2916.136633


We load all five World Bank files, take the 2023 value from each, and combine them
into one table using the country code. We keep the country code so we can later
drop regional aggregates and match with the other sources.

In [18]:
wb_files = {
    "gdp_per_capita":     "API_NY_GDP_PCAP_CD_DS2_en_csv_v2_358064.csv",
    "health_expenditure": "API_SH_XPD_CHEX_GD_ZS_DS2_en_csv_v2_115482.csv",
    "unemployment":       "API_SL_UEM_TOTL_ZS_DS2_en_csv_v2_375762.csv",
    "inflation":          "API_FP_CPI_TOTL_ZG_DS2_en_csv_v2_369762.csv",
    "population":         "API_SP_POP_TOTL_DS2_en_csv_v2_346039.csv",
}

worldbank = None

for name, fname in wb_files.items():
    df = pd.read_csv(DATA_RAW / fname, skiprows=4)
    df = df[["Country Name", "Country Code", "2023"]].rename(
        columns={"Country Name": "country", "Country Code": "code", "2023": name}
    )
    if worldbank is None:
        worldbank = df
    else:
        worldbank = worldbank.merge(df[["code", name]], on="code", how="outer")

print("Shape:", worldbank.shape)
print(worldbank.isna().sum())
worldbank.head()

Shape: (266, 7)
country                0
code                   0
gdp_per_capita        15
health_expenditure    26
unemployment          34
inflation             88
population             1
dtype: int64


,country,code,gdp_per_capita,health_expenditure,unemployment,inflation,population
0,Aruba,ABW,35718.753119,NaN,NaN,NaN,107359.0
1,Africa Eastern and Southern,AFE,1571.449189,5.540810,7.676548,7.399186,750491370.0
2,Afghanistan,AFG,413.757895,14.985763,14.008000,-4.644709,41454761.0
3,Africa Western and Central,AFW,1841.855064,4.143225,3.200801,5.221168,509398589.0
4,Angola,AGO,2916.136633,2.546929,14.051000,13.644102,36749906.0


## 5. Corruption (CPI) data

Source: Transparency International, Corruption Perceptions Index 2025.

This file is saved in a strict Excel format that pandas may not read directly. We
first try a normal read. If it fails, we convert the file once and read the
converted version.

In [19]:
cpi_file = DATA_RAW / "CPI2025_Results.xlsx"

try:
    test = pd.ExcelFile(cpi_file)
    print("Sheets found:", test.sheet_names)
except Exception as e:
    print("Could not read directly:", repr(e))

Sheets found: []


C:\Users\ASHUTOSH\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:84: UserWarning: File contains an invalid specification for 0. This will be removed
  warn(msg)
C:\Users\ASHUTOSH\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:102: UserWarning: Defined names for sheet index 4 cannot be located
  warn(f"Defined names for sheet index {idx} cannot be located")
C:\Users\ASHUTOSH\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:102: UserWarning: Defined names for sheet index 5 cannot be located
  warn(f"Defined names for sheet index {idx} cannot be located")
C:\Users\ASHUTOSH\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\workbook.py:102: UserWarning: Defined names for sheet index 1 cannot be located
  warn(f"Defined names for sheet index {idx} cannot be located")
C:\Users\ASHUTOSH\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\read

The CPI file is in a strict Excel format that pandas cannot read. We open it with
openpyxl and save a normal-format copy in the same folder, then read the copy.

In [20]:
import openpyxl

cpi_fixed = DATA_RAW / "CPI2025_Results_fixed.xlsx"

wb = openpyxl.load_workbook(cpi_file)
wb.save(cpi_fixed)

cpi_check = pd.ExcelFile(cpi_fixed)
print("Sheets found:", cpi_check.sheet_names)

IndexError: At least one sheet must be visible

The CPI file is in a strict Excel format. A strict xlsx is a zip of XML files that
use a different namespace, which pandas cannot read. We rewrite the namespaces to
the standard ones, repackage the file, and then read the normal copy.

In [21]:
import zipfile, shutil, os

def fix_strict_xlsx(src, out):
    work = str(src) + "_unzipped"
    if os.path.exists(work):
        shutil.rmtree(work)
    os.makedirs(work)

    with zipfile.ZipFile(src) as z:
        z.extractall(work)

    replacements = {
        "http://purl.oclc.org/ooxml/spreadsheetml/main":
            "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "http://purl.oclc.org/ooxml/officeDocument/relationships":
            "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "http://purl.oclc.org/ooxml/officeDocument/2006/relationships":
            "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "http://purl.oclc.org/ooxml/package/2006/content-types":
            "http://schemas.openxmlformats.org/package/2006/content-types",
    }

    for root, _, files in os.walk(work):
        for f in files:
            if f.endswith((".xml", ".rels")):
                p = os.path.join(root, f)
                with open(p, "r", encoding="utf-8") as fh:
                    data = fh.read()
                for old, new in replacements.items():
                    data = data.replace(old, new)
                with open(p, "w", encoding="utf-8") as fh:
                    fh.write(data)

    if os.path.exists(out):
        os.remove(out)
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(work):
            for f in files:
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, work))

    shutil.rmtree(work)


cpi_fixed = DATA_RAW / "CPI2025_Results_fixed.xlsx"
fix_strict_xlsx(cpi_file, cpi_fixed)

cpi_check = pd.ExcelFile(cpi_fixed)
print("Sheets found:", cpi_check.sheet_names)

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'D:\\beyond-gdp-capability-analysis\\data\\raw\\CPI2025_Results_fixed.xlsx'

In [22]:
import zipfile, shutil, os

def fix_strict_xlsx(src, out):
    work = str(src) + "_unzipped"
    if os.path.exists(work):
        shutil.rmtree(work)
    os.makedirs(work)

    with zipfile.ZipFile(src) as z:
        z.extractall(work)

    replacements = {
        "http://purl.oclc.org/ooxml/spreadsheetml/main":
            "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "http://purl.oclc.org/ooxml/officeDocument/relationships":
            "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "http://purl.oclc.org/ooxml/officeDocument/2006/relationships":
            "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "http://purl.oclc.org/ooxml/package/2006/content-types":
            "http://schemas.openxmlformats.org/package/2006/content-types",
    }

    for root, _, files in os.walk(work):
        for f in files:
            if f.endswith((".xml", ".rels")):
                p = os.path.join(root, f)
                with open(p, "r", encoding="utf-8") as fh:
                    data = fh.read()
                for old, new in replacements.items():
                    data = data.replace(old, new)
                with open(p, "w", encoding="utf-8") as fh:
                    fh.write(data)

    if os.path.exists(out):
        os.remove(out)
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(work):
            for f in files:
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, work))

    shutil.rmtree(work)


cpi_fixed = DATA_RAW / "CPI2025_Results_clean.xlsx"
fix_strict_xlsx(cpi_file, cpi_fixed)

cpi_check = pd.ExcelFile(cpi_fixed)
print("Sheets found:", cpi_check.sheet_names)

Sheets found: ['CPI2025', 'CPI Timeseries 2012 - 2025', 'CPI 2025 Regional Averages', 'Significant Changes since 2012 ', 'CPI 2025 Trends', 'CPI Historical']


We read the first sheet, CPI2025. The top rows are title and notes, so the real
header is on row 4 (header=3). We keep the country, ISO3 code, and the 2025 score.

In [23]:
cpi_raw = pd.read_excel(cpi_fixed, sheet_name="CPI2025", header=3)

print("Shape:", cpi_raw.shape)
print(cpi_raw.columns[:7].tolist())
cpi_raw.head()

Shape: (182, 22)
['Country / Territory', 'ISO3', 'Region', 'CPI 2025 score', 'Rank', 'Standard error 2025', 'Number of sources']


,Country / Territory,ISO3,Region,CPI 2025 score,Rank,Standard error 2025,Number of sources,Lower CI,Upper CI,African Development Bank CPIA,...,Economist Intelligence Unit Country Ratings,Freedom House Nations In Transit,Global Insights Country Risk Ratings,IMD World Competitiveness Yearbook,PERC Asia Risk Guide,PRS International Country Risk Guide,Varieties of Democracy Project,World Bank CPIA,World Economic Forum EOS,World Justice Project Rule of Law Index
0,Denmark,DNK,WE/EU,89,1,1.882284,8,85.91306,92.08694,NaN,...,82.83861,NaN,85.22919,93.08286,NaN,100.00000,75.12891,NaN,93.58622,87.02797
1,Finland,FIN,WE/EU,88,2,1.806286,8,85.03769,90.96231,NaN,...,82.83861,NaN,85.22919,93.08286,NaN,96.30098,73.75974,NaN,90.56377,84.42215
2,Singapore,SGP,AP,84,3,1.518263,9,81.51005,86.48995,NaN,...,82.83861,NaN,85.22919,90.84637,88.35136,78.14610,73.07516,NaN,99.16616,85.29076
3,New Zealand,NZL,AP,81,4,2.036385,8,77.66033,84.33967,NaN,...,82.83861,NaN,85.22919,72.44612,NaN,96.30098,74.44433,NaN,68.94151,82.68494
4,Norway,NOR,WE/EU,81,4,1.618172,8,78.34620,83.65380,NaN,...,82.83861,NaN,85.22919,78.64731,NaN,87.22354,74.44433,NaN,67.08154,84.42215


We keep three columns: country, ISO3 code, and the 2025 CPI score. The score runs
from 0 (highly corrupt) to 100 (very clean). The code lets us match this with the
other sources later.

In [24]:
cpi = cpi_raw[["Country / Territory", "ISO3", "CPI 2025 score"]].rename(
    columns={"Country / Territory": "country", "ISO3": "code", "CPI 2025 score": "cpi_score"}
)
cpi = cpi[cpi["code"].notna()].reset_index(drop=True)

print("Shape:", cpi.shape)
print(cpi.isna().sum())
cpi.head()

Shape: (182, 3)
country      0
code         0
cpi_score    0
dtype: int64


,country,code,cpi_score
0,Denmark,DNK,89
1,Finland,FIN,88
2,Singapore,SGP,84
3,New Zealand,NZL,81
4,Norway,NOR,81


## 6. Coverage summary

A short summary of each source: number of rows and number of columns. This closes
the data understanding stage before we move to cleaning and merging.

In [25]:
sources = {
    "HDI": hdi,
    "Happiness": happiness,
    "World Bank": worldbank,
    "CPI": cpi,
}

for name, df in sources.items():
    print(f"{name:12s} rows: {df.shape[0]:>4}   columns: {df.shape[1]}")

HDI          rows:  193   columns: 7
Happiness    rows:  143   columns: 6
World Bank   rows:  266   columns: 7
CPI          rows:  182   columns: 3


In [26]:
import country_converter as coco

cc = coco.CountryConverter()

test = cc.convert(["India", "South Korea", "Korea (Republic of)"], to="ISO3")
print(test)

['IND', 'KOR', 'KOR']


## Add ISO3 codes to HDI and Happiness

World Bank and CPI already have ISO3 codes. HDI and Happiness only have country
names, so we convert those names to ISO3 codes using country_converter. We then
check whether any name failed to convert.

In [27]:
hdi["code"] = cc.convert(hdi["country"].tolist(), to="ISO3")
happiness["code"] = cc.convert(happiness["country"].tolist(), to="ISO3")

print("HDI not found:", (hdi["code"] == "not found").sum())
print("Happiness not found:", (happiness["code"] == "not found").sum())

hdi[["country", "code"]].head()

HDI not found: 0
Happiness not found: 0


,country,code
0,Iceland,ISL
1,Norway,NOR
2,Switzerland,CHE
3,Denmark,DNK
4,Germany,DEU


## Remove aggregates from World Bank data

The World Bank file includes regional and income-group aggregates (for example
"Africa Eastern and Southern"), not just countries. We keep only rows whose ISO3
code maps to a real country and drop the aggregates.

In [28]:
wb_names = cc.convert(worldbank["code"].tolist(), to="name_short")
worldbank["valid_country"] = [n != "not found" for n in wb_names]

print("Total rows:", len(worldbank))
print("Real countries:", worldbank["valid_country"].sum())
print("Aggregates removed:", (~worldbank["valid_country"]).sum())

worldbank_countries = worldbank[worldbank["valid_country"]].drop(columns="valid_country").reset_index(drop=True)
print("Shape after:", worldbank_countries.shape)

AFE not found in ISO3
AFW not found in ISO3
ARB not found in ISO3
CEB not found in ISO3
CHI not found in ISO3
CSS not found in ISO3
EAP not found in ISO3
EAR not found in ISO3
EAS not found in ISO3
ECA not found in ISO3
ECS not found in ISO3
EMU not found in ISO3
EUU not found in ISO3
FCS not found in ISO3
HIC not found in ISO3
HPC not found in ISO3
IBD not found in ISO3
IBT not found in ISO3
IDA not found in ISO3
IDB not found in ISO3
IDX not found in ISO3
INX not found in ISO3
LAC not found in ISO3
LCN not found in ISO3
LDC not found in ISO3
LIC not found in ISO3
LMC not found in ISO3
LMY not found in ISO3
LTE not found in ISO3
MEA not found in ISO3
MIC not found in ISO3
MNA not found in ISO3
NAC not found in ISO3
OED not found in ISO3
OSS not found in ISO3
PRE not found in ISO3
PSS not found in ISO3
PST not found in ISO3
SAS not found in ISO3
SSA not found in ISO3
SSF not found in ISO3
SST not found in ISO3
TEA not found in ISO3
TEC not found in ISO3
TLA not found in ISO3
TMN not fo

Total rows: 266
Real countries: 216
Aggregates removed: 50
Shape after: (216, 7)


## Merge all four sources

We merge the four tables on the ISO3 code using inner joins, so only countries
present in all four sources are kept. This gives a complete dataset with no gaps,
which is better for the analysis that follows. The cost is fewer countries, mainly
because the Happiness data covers fewer countries.

In [29]:
hdi_m = hdi.drop(columns="hdi_rank")
happ_m = happiness.drop(columns="country")
wb_m = worldbank_countries.drop(columns="country")
cpi_m = cpi.drop(columns="country")

master = (
    hdi_m
    .merge(happ_m, on="code", how="inner")
    .merge(wb_m, on="code", how="inner")
    .merge(cpi_m, on="code", how="inner")
)

master = master.reset_index(drop=True)

print("Shape:", master.shape)
print("Countries:", master["country"].nunique())
master.head()

Shape: (140, 18)
Countries: 140


,country,hdi,life_expectancy,expected_schooling,mean_schooling,gni_per_capita,code,happiness_score,social_support_contrib,freedom_contrib,generosity_contrib,corruption_contrib,gdp_per_capita,health_expenditure,unemployment,inflation,population,cpi_score
0,Iceland,0.972,82.691,18.850590,13.908926,69116.93736,ISL,7.525,1.617,0.819,0.258,0.182,82138.789297,8.705101,3.518,8.736303,385663.0,77
1,Norway,0.970,83.308,18.792850,13.117962,112710.02110,NOR,7.302,1.517,0.835,0.224,0.484,87497.217965,9.429255,3.574,5.517850,5519601.0,81
2,Switzerland,0.970,83.954,16.667530,13.949121,81948.90177,CHE,7.060,1.425,0.759,0.173,0.498,100623.549627,11.690961,4.043,2.135401,8888822.0,80
3,Denmark,0.962,81.933,18.704010,13.027321,76007.85669,DNK,7.583,1.520,0.823,0.204,0.548,68043.546697,9.557296,5.094,3.305178,5946952.0,89
4,Germany,0.959,81.378,17.309219,14.296372,64053.22124,DEU,6.719,1.390,0.700,0.174,0.368,54776.766824,11.744287,3.071,5.946437,83287273.0,77


## Final master dataset

The four sources are now merged into one country-level table for 2023. We reorder
the columns so the identifiers come first, and confirm there are no missing values
before saving.

In [31]:
order = [
    "country", "code",
    "hdi", "life_expectancy", "expected_schooling", "mean_schooling", "gni_per_capita",
    "happiness_score", "social_support_contrib", "freedom_contrib",
    "generosity_contrib", "corruption_contrib",
    "gdp_per_capita", "health_expenditure", "unemployment", "inflation", "population",
    "cpi_score",
]

master = master[order]

print("Shape:", master.shape)
print("Total missing values:", master.isna().sum().sum())
print(master.isna().sum())

Shape: (140, 18)
Total missing values: 19
country                   0
code                      0
hdi                       0
life_expectancy           0
expected_schooling        0
mean_schooling            0
gni_per_capita            0
happiness_score           0
social_support_contrib    2
freedom_contrib           2
generosity_contrib        2
corruption_contrib        2
gdp_per_capita            1
health_expenditure        2
unemployment              1
inflation                 7
population                0
cpi_score                 0
dtype: int64


In [32]:
rows_with_missing = master[master.isna().any(axis=1)]
print("Countries with any missing value:", len(rows_with_missing))
rows_with_missing[["country", "inflation", "unemployment", "gdp_per_capita",
                   "health_expenditure", "social_support_contrib"]]

Countries with any missing value: 10


,country,inflation,unemployment,gdp_per_capita,health_expenditure,social_support_contrib
7,"Hong Kong, China (SAR)",2.096605,2.949,50562.363314,NaN,1.184
34,Bahrain,0.074621,1.049,29290.128422,3.995624,NaN
70,Ukraine,12.849022,NaN,5139.598145,NaN,1.315
93,Venezuela (Bolivarian Republic of),NaN,5.407,3617.435946,3.776114,1.321
95,Eswatini (Kingdom of),NaN,34.610,3755.521112,7.497640,0.925
97,Tajikistan,NaN,7.037,1178.479901,7.437899,NaN
109,Myanmar,NaN,2.977,1233.196662,4.457484,0.988
112,Zimbabwe,NaN,9.348,2195.224921,2.926016,0.850
125,Congo (Democratic Republic of the),NaN,4.469,660.212052,3.684951,0.665
134,Yemen,NaN,16.944,NaN,9.695592,1.281


## Handle missing values

Ten countries have one or more missing values, mostly in inflation, for economies
where such data is hard to report (for example Venezuela, Ukraine, Yemen). We drop
these incomplete rows so the analysis runs on complete cases. This keeps 130
countries with no missing values.

In [33]:
master = master.dropna().reset_index(drop=True)

print("Shape:", master.shape)
print("Total missing values:", master.isna().sum().sum())
print("Countries:", master["country"].nunique())

Shape: (130, 18)
Total missing values: 0
Countries: 130


## Save the master dataset

We save the cleaned dataset to data/processed/master_dataset.csv. The later
notebooks (database, SQL, analysis) load this file directly instead of repeating
the cleaning steps.

In [34]:
processed_dir = Path(r"D:\beyond-gdp-capability-analysis\data\processed")
master_path = processed_dir / "master_dataset.csv"

master.to_csv(master_path, index=False)

print("Saved to:", master_path)
print("File exists:", master_path.exists())

Saved to: D:\beyond-gdp-capability-analysis\data\processed\master_dataset.csv
File exists: True
